In [0]:
# POC-2: Retail Order CDC & Revenue Lakehouse
# Phase 2: Bronze Layer Ingestion

from pyspark.sql import functions as F
from pyspark.sql import types as T
from delta.tables import DeltaTable

# Project configuration
CATALOG = "retail_poc2"
BRONZE_SCHEMA = "bronze"

BASE_PATH = "/Volumes/retail_poc2/landing/source_files"

INITIAL_PATH = f"{BASE_PATH}/initial"
CDC_PATH = f"{BASE_PATH}/CDC"

# Target tables
CUSTOMERS_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.customers_raw"
PRODUCTS_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.products_raw"
ORDERS_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.orders_raw"
CDC_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.orders_cdc_raw"

print("BRONZE PIPELINE CONFIGURATION")
print("-" * 50)

print("Spark version:", spark.version)
print("Initial source:", INITIAL_PATH)
print("CDC source:", CDC_PATH)

print("\nTarget tables:")
print(CUSTOMERS_TABLE)
print(PRODUCTS_TABLE)
print(ORDERS_TABLE)
print(CDC_TABLE)

print("\nCONFIGURATION COMPLETED")

In [0]:

# Step 2.2: Explicit source schemas

# Customer schema
customers_schema = T.StructType([
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("email", T.StringType(), True),
    T.StructField("city", T.StringType(), True),
    T.StructField("state", T.StringType(), True),
    T.StructField("created_at", T.TimestampType(), True)
])

# Product schema
products_schema = T.StructType([
    T.StructField("product_id", T.StringType(), True),
    T.StructField("product_name", T.StringType(), True),
    T.StructField("category", T.StringType(), True),
    T.StructField("unit_price", T.DecimalType(18, 2), True),
    T.StructField("active_flag", T.BooleanType(), True)
])

# Initial order schema
orders_schema = T.StructType([
    T.StructField("order_id", T.StringType(), True),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("product_id", T.StringType(), True),
    T.StructField("order_date", T.DateType(), True),
    T.StructField("quantity", T.IntegerType(), True),
    T.StructField("unit_price", T.DecimalType(18, 2), True),
    T.StructField("order_status", T.StringType(), True),
    T.StructField("updated_at", T.TimestampType(), True),
    T.StructField("source_sequence", T.LongType(), True)
])

# CDC event schema
cdc_schema = T.StructType([
    T.StructField("event_id", T.StringType(), True),
    T.StructField("order_id", T.StringType(), True),
    T.StructField("operation", T.StringType(), True),
    T.StructField("event_sequence", T.LongType(), True),
    T.StructField("event_timestamp", T.TimestampType(), True),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("product_id", T.StringType(), True),
    T.StructField("order_date", T.DateType(), True),
    T.StructField("quantity", T.IntegerType(), True),
    T.StructField("unit_price", T.DecimalType(18, 2), True),
    T.StructField("order_status", T.StringType(), True),
    T.StructField("updated_at", T.TimestampType(), True)
])

print("EXPLICIT SOURCE SCHEMAS")
print("-" * 50)

schemas = {
    "Customers": customers_schema,
    "Products": products_schema,
    "Orders": orders_schema,
    "CDC Events": cdc_schema
}

for name, schema in schemas.items():
    print(f"\n{name}: {len(schema.fields)} columns")
    print(schema.simpleString())

print("\nSCHEMA DEFINITION COMPLETED")


In [0]:

# Step 2.3: Source reading and ingestion metadata

def read_source(path, schema):

    # Read source as strings to inspect price precision
    raw_df = (
        spark.read
        .option("header", True)
        .option("mode", "FAILFAST")
        .csv(path)
    )

    # Validate source column names and order
    if raw_df.columns != schema.fieldNames():
        raise ValueError(f"Source header mismatch: {path}")

    # Prevent silent rounding of financial values
    if "unit_price" in raw_df.columns:
        invalid_prices = raw_df.filter(
            F.col("unit_price").isNotNull()
            & ~F.trim(F.col("unit_price")).rlike(
                r"^[+-]?\d{1,16}(\.\d{1,2})?$"
            )
        )

        if invalid_prices.limit(1).count() > 0:
            raise ValueError(
                f"Invalid price format or precision: {path}"
            )

    # Read using the explicit schema
    df = (
        spark.read
        .option("header", True)
        .option("mode", "FAILFAST")
        .option("enforceSchema", False)
        .schema(schema)
        .csv(path)
    )

    # Add technical ingestion metadata
    return df.select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
        F.col("_metadata.file_name").alias("_source_file_name"),
        F.current_timestamp().alias("_ingestion_timestamp"),
        F.sha2(
            F.to_json(
                F.struct(*[F.col(c) for c in schema.fieldNames()]),
                {"ignoreNullFields": "false"}
            ),
            256
        ).alias("_record_hash")
    )


# Read all four source datasets
customers_df = read_source(
    f"{INITIAL_PATH}/customers.csv",
    customers_schema
)

products_df = read_source(
    f"{INITIAL_PATH}/products.csv",
    products_schema
)

orders_df = read_source(
    f"{INITIAL_PATH}/orders.csv",
    orders_schema
)

cdc_df = read_source(
    f"{CDC_PATH}/orders_cdc_batch_*.csv",
    cdc_schema
)

# Source reconciliation
datasets = [
    ("Customers", customers_df, 100),
    ("Products", products_df, 30),
    ("Orders", orders_df, 500),
    ("CDC Events", cdc_df, 15)
]

print("BRONZE SOURCE RECONCILIATION")
print("-" * 50)

for name, df, expected in datasets:

    actual = df.count()

    print(f"{name}: {actual} records")

    assert actual == expected, (
        f"{name}: Expected {expected}, found {actual}"
    )

print("\nCDC FILE-LEVEL COUNTS")

cdc_df.groupBy("_source_file_name").count().orderBy(
    "_source_file_name"
).show(truncate=False)

print("\nSAMPLE INGESTION METADATA")

cdc_df.select(
    "event_id",
    "order_id",
    "_source_file_name",
    "_ingestion_timestamp",
    "_record_hash"
).show(5, truncate=False)

print("\nSOURCE READING AND METADATA COMPLETED")


In [0]:

# Step 2.4: Replay-safe Bronze Delta ingestion

def ingest_bronze(df, target_table):

    # Validate ingestion identity
    identity_columns = [
        "_source_file",
        "_record_hash"
    ]

    source_count = df.count()

    unique_count = (
        df.select(*identity_columns)
        .distinct()
        .count()
    )

    # Prevent ambiguous duplicate records
    if source_count != unique_count:
        raise ValueError(
            f"Duplicate ingestion identities found: {target_table}"
        )

    # Create an empty Delta table if it does not exist
    if not spark.catalog.tableExists(target_table):

        (
            df.limit(0)
            .write
            .format("delta")
            .mode("ignore")
            .saveAsTable(target_table)
        )

        print(f"Created table: {target_table}")

    # Count existing records
    before_count = spark.table(target_table).count()

    # Reference the Delta target
    target = DeltaTable.forName(spark, target_table)

    # Insert only previously unseen records
    (
        target.alias("t")
        .merge(
            df.alias("s"),
            """
            t._source_file = s._source_file
            AND t._record_hash = s._record_hash
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    # Count records after ingestion
    after_count = spark.table(target_table).count()

    inserted_count = after_count - before_count

    print(f"Target: {target_table}")
    print(f"Source records: {source_count}")
    print(f"Before ingestion: {before_count}")
    print(f"Inserted records: {inserted_count}")
    print(f"After ingestion: {after_count}")
    print("-" * 50)


# Execute Bronze ingestion

print("BRONZE DELTA INGESTION")
print("=" * 50)

ingest_bronze(customers_df, CUSTOMERS_TABLE)
ingest_bronze(products_df, PRODUCTS_TABLE)
ingest_bronze(orders_df, ORDERS_TABLE)
ingest_bronze(cdc_df, CDC_TABLE)

print("BRONZE INGESTION COMPLETED")


In [0]:

# Step 2.5: Bronze data reconciliation

def validate_bronze(source_df, target_table):

    target_df = spark.table(target_table)

    source_count = source_df.count()
    target_count = target_df.count()

    # Exclude ingestion timestamp because it is
    # generated independently during query execution
    compare_columns = [
        c for c in source_df.columns
        if c != "_ingestion_timestamp"
    ]

    # Check missing source records
    missing_count = (
        source_df.select(*compare_columns)
        .exceptAll(target_df.select(*compare_columns))
        .count()
    )

    # Check unexpected target records
    unexpected_count = (
        target_df.select(*compare_columns)
        .exceptAll(source_df.select(*compare_columns))
        .count()
    )

    # Check duplicate ingestion identities
    duplicate_count = (
        target_df
        .groupBy("_source_file", "_record_hash")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    # Validate required metadata
    invalid_metadata = (
        target_df.filter(
            F.col("_source_file").isNull()
            | F.col("_source_file_name").isNull()
            | F.col("_ingestion_timestamp").isNull()
            | F.col("_record_hash").isNull()
            | (F.length("_record_hash") != 64)
        )
        .count()
    )

    # Verify Delta storage format
    table_detail = spark.sql(
        f"DESCRIBE DETAIL {target_table}"
    ).first()

    storage_format = table_detail["format"]

    # Verify managed table type
    table_type = spark.catalog.getTable(
        target_table
    ).tableType

    passed = (
        source_count == target_count
        and missing_count == 0
        and unexpected_count == 0
        and duplicate_count == 0
        and invalid_metadata == 0
        and storage_format.lower() == "delta"
        and table_type.upper() == "MANAGED"
    )

    print(f"\nTable: {target_table}")
    print(f"Source records: {source_count}")
    print(f"Target records: {target_count}")
    print(f"Missing records: {missing_count}")
    print(f"Unexpected records: {unexpected_count}")
    print(f"Duplicate identity groups: {duplicate_count}")
    print(f"Invalid metadata records: {invalid_metadata}")
    print(f"Storage format: {storage_format}")
    print(f"Table type: {table_type}")
    print(f"Result: {'PASS' if passed else 'FAIL'}")
    print("-" * 50)

    return passed


print("BRONZE DATA RECONCILIATION")
print("=" * 50)

results = [
    validate_bronze(customers_df, CUSTOMERS_TABLE),
    validate_bronze(products_df, PRODUCTS_TABLE),
    validate_bronze(orders_df, ORDERS_TABLE),
    validate_bronze(cdc_df, CDC_TABLE)
]

print("\nRECONCILIATION SUMMARY")
print(f"Tables validated: {len(results)}")
print(f"Passed: {sum(results)}")
print(f"Failed: {len(results) - sum(results)}")

assert all(results), "Bronze reconciliation failed"

print("BRONZE RECONCILIATION PASSED")


In [0]:

# Step 2.6: Bronze idempotency testing

test_datasets = [
    (customers_df, CUSTOMERS_TABLE),
    (products_df, PRODUCTS_TABLE),
    (orders_df, ORDERS_TABLE),
    (cdc_df, CDC_TABLE)
]


def test_idempotency(source_df, target_table):

    # Capture existing Delta table version
    before_version = (
        DeltaTable.forName(spark, target_table)
        .history(1)
        .select("version")
        .first()["version"]
    )

    # Read the original table snapshot
    before_df = (
        spark.read
        .format("delta")
        .option("versionAsOf", before_version)
        .table(target_table)
    )

    before_count = before_df.count()

    # Reprocess the same source data
    ingest_bronze(source_df, target_table)

    # Read the latest table state
    after_df = spark.table(target_table)

    after_count = after_df.count()

    # Compare complete records, including metadata
    removed_records = (
        before_df.exceptAll(after_df).count()
    )

    added_records = (
        after_df.exceptAll(before_df).count()
    )

    passed = (
        before_count == after_count
        and removed_records == 0
        and added_records == 0
    )

    print(f"IDEMPOTENCY TEST: {target_table}")
    print(f"Original Delta version: {before_version}")
    print(f"Before rerun: {before_count}")
    print(f"After rerun: {after_count}")
    print(f"Removed or changed records: {removed_records}")
    print(f"Added or changed records: {added_records}")
    print(f"Result: {'PASS' if passed else 'FAIL'}")
    print("=" * 50)

    return passed


print("BRONZE IDEMPOTENCY TESTING")
print("=" * 50)

idempotency_results = []

for source_df, target_table in test_datasets:

    result = test_idempotency(
        source_df,
        target_table
    )

    idempotency_results.append(result)


print("\nIDEMPOTENCY TEST SUMMARY")
print(f"Tables tested: {len(idempotency_results)}")
print(f"Passed: {sum(idempotency_results)}")
print(
    f"Failed: "
    f"{len(idempotency_results) - sum(idempotency_results)}"
)

assert all(idempotency_results), "Idempotency test failed"

print("BRONZE IDEMPOTENCY TEST PASSED")
